# 02 — Clean SSP Forecast Data

**Goal:** Load SSP scenario forecasts for all features, harmonize country names, merge into one panel.  
**Reads:**
- `data/raw/GDP(Forecast)_POP_SSP_1950_2100.xlsx` (GDP & Population)
- `data/raw/ControlOfCorruption_Forecast_SSPExtensionExplorer_2015-2099.csv`
- `data/raw/EmploymentInAgriculture_Forecast_SSPExtensionExplorer_2016-2050.csv`
- `data/raw/GiniCoefficient_Forecast_SSPExtensionExplorer_2015-2100.csv`
- `data/raw/HumanDevelopmentIndex_Forecast_SSPExtensionExplorer_2010-2075.csv`

**Outputs:** `data/processed/ssp_forecast_panel.csv`

Scenarios kept: **SSP1, SSP4, SSP5**  
Time steps: **5-year intervals** (2025, 2030, …, 2100)  

## Imports and paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
OUTPUTS_DIR = Path("../outputs")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = ["SSP1", "SSP4", "SSP5"]

# 5-year steps we want in the final output
FIVE_YEAR_STEPS = list(range(2025, 2101, 5))
print(f"Target years: {FIVE_YEAR_STEPS[:5]} ... {FIVE_YEAR_STEPS[-3:]}")
print(f"Scenarios: {SCENARIOS}")

## 1. Load GDP & Population from SSP Excel

The Excel file has columns: `Model, Scenario, Region, Variable, Unit`, then 5-year year columns.  
We extract `GDP|PPP`, `Population`, and `GDP|PPP [per capita]` for SSP1/4/5.

In [ ]:
ssp_raw = pd.read_excel(
    DATA_RAW / "GDP(Forecast)_POP_SSP_1950_2100.xlsx",
    sheet_name="data"
)
print(f"SSP raw shape: {ssp_raw.shape}")
print(f"Variables: {ssp_raw['Variable'].unique()[:10]}")
print(f"Scenarios: {ssp_raw['Scenario'].unique()}")
ssp_raw.head(3)

In [ ]:
# Filter to our scenarios and key variables
KEEP_VARS = ["GDP|PPP", "Population", "GDP|PPP [per capita]"]
ssp_filtered = ssp_raw[
    (ssp_raw["Scenario"].isin(SCENARIOS)) &
    (ssp_raw["Variable"].isin(KEEP_VARS))
].copy()

# Remove regional aggregates (contain parentheses like "(R10)")
ssp_filtered = ssp_filtered[~ssp_filtered["Region"].str.contains(r"\(", na=False)]

print(f"After filtering: {ssp_filtered.shape}")
print(f"Variables: {ssp_filtered['Variable'].unique()}")
print(f"Regions: {ssp_filtered['Region'].nunique()}")

# Print units for reference
for var in KEEP_VARS:
    subset = ssp_filtered[ssp_filtered["Variable"] == var]
    if len(subset) > 0:
        print(f"  {var}: unit = {subset['Unit'].iloc[0]}, rows = {len(subset)}")

### Melt to long format and pivot variables to columns

In [ ]:
# Identify year columns (numeric columns)
year_cols = [c for c in ssp_filtered.columns if isinstance(c, (int, float))]
print(f"Year columns: {year_cols[:5]} ... {year_cols[-3:]}")

# Melt to long
ssp_long = ssp_filtered.melt(
    id_vars=["Scenario", "Region", "Variable"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)
ssp_long["year"] = ssp_long["year"].astype(int)
ssp_long["value"] = pd.to_numeric(ssp_long["value"], errors="coerce")

# Keep only 5-year steps from 2025 onward
ssp_long = ssp_long[ssp_long["year"].isin(FIVE_YEAR_STEPS)]

print(f"SSP long shape: {ssp_long.shape}")
ssp_long.head()

In [ ]:
# Pivot: one column per variable
VAR_RENAME = {
    "GDP|PPP": "gdp",
    "Population": "population",
    "GDP|PPP [per capita]": "gdp_per_capita"
}
ssp_long["Variable"] = ssp_long["Variable"].map(VAR_RENAME)

ssp_pivot = ssp_long.pivot_table(
    index=["Region", "Scenario", "year"],
    columns="Variable",
    values="value"
).reset_index()

ssp_pivot.columns.name = None
ssp_pivot = ssp_pivot.rename(columns={"Region": "country_name", "Scenario": "scenario"})

# If gdp_per_capita was not in the file, compute it
if "gdp_per_capita" not in ssp_pivot.columns or ssp_pivot["gdp_per_capita"].isna().all():
    print("GDP per capita not available — computing from GDP / Population")
    # GDP is in billion USD, population is in million → per capita in thousands USD
    ssp_pivot["gdp_per_capita"] = ssp_pivot["gdp"] / ssp_pivot["population"]

print(f"SSP GDP/Pop pivot: {ssp_pivot.shape}")
print(f"Countries: {ssp_pivot['country_name'].nunique()}")
ssp_pivot.head()

## 2. Load SSP Extension Explorer forecasts

These CSVs share a format: `model, scenario, region, unit, variable`, then year columns.  
We filter to SSP1/4/5 and resample to 5-year steps.

In [ ]:
def load_ssp_explorer(filepath, var_name):
    """
    Load an SSP Extension Explorer forecast CSV.
    Filters to SSP1/4/5, keeps 5-year steps, melts to long format.
    Returns: DataFrame with country_name, scenario, year, <var_name>
    """
    df = pd.read_csv(filepath)
    print(f"  Raw shape: {df.shape}")
    print(f"  Variable: {df['variable'].iloc[0]}")
    
    # Filter scenarios
    df = df[df["scenario"].isin(SCENARIOS)].copy()
    
    # Identify year columns (string digits)
    year_cols_all = [c for c in df.columns if c.isdigit()]
    
    # Keep only 5-year step columns that exist
    year_cols_5yr = [c for c in year_cols_all if int(c) in FIVE_YEAR_STEPS]
    print(f"  5-year columns available: {year_cols_5yr[:5]} ... {year_cols_5yr[-3:] if len(year_cols_5yr) > 3 else year_cols_5yr}")
    
    # Melt
    df = df.melt(
        id_vars=["scenario", "region"],
        value_vars=year_cols_5yr,
        var_name="year",
        value_name=var_name
    )
    df["year"] = df["year"].astype(int)
    df[var_name] = pd.to_numeric(df[var_name], errors="coerce")
    df = df.rename(columns={"region": "country_name"})
    
    print(f"  Result shape: {df.shape}, Countries: {df['country_name'].nunique()}")
    print(f"  Year range: {df['year'].min()}–{df['year'].max()}")
    return df

### Load Control of Corruption forecast

In [ ]:
print("Loading Control of Corruption forecast...")
corruption_fc = load_ssp_explorer(
    DATA_RAW / "ControlOfCorruption_Forecast_SSPExtensionExplorer_2015-2099.csv",
    "control_of_corruption"
)
corruption_fc.head()

### Load Employment in Agriculture forecast

This file only goes to **2050**. We'll extrapolate 2055–2100 later.

In [ ]:
print("Loading Employment in Agriculture forecast...")
agri_fc = load_ssp_explorer(
    DATA_RAW / "EmploymentInAgriculture_Forecast_SSPExtensionExplorer_2016-2050.csv",
    "employment_agriculture"
)
agri_fc.head()

### Load Gini Coefficient forecast

In [ ]:
print("Loading Gini Coefficient forecast...")
gini_fc = load_ssp_explorer(
    DATA_RAW / "GiniCoefficient_Forecast_SSPExtensionExplorer_2015-2100.csv",
    "gini"
)
gini_fc.head()

### Load HDI forecast

This file only goes to **2075**. We'll extrapolate 2080–2100 later.

In [ ]:
print("Loading HDI forecast...")
hdi_fc = load_ssp_explorer(
    DATA_RAW / "HumanDevelopmentIndex_Forecast_SSPExtensionExplorer_2010-2075.csv",
    "hdi"
)
hdi_fc.head()

## 3. Extrapolate short-horizon variables

- **Employment in Agriculture**: 2050 → extrapolate to 2100 using linear trend of last 3 data points
- **HDI**: 2075 → extrapolate to 2100 using linear trend of last 3 data points

Extrapolated values are flagged with `is_extrapolated = True`.

In [ ]:
def extrapolate_linear(df, var_name, max_year_in_data, target_max_year, clip_min, clip_max):
    """
    Extrapolate beyond the data's last year using a linear trend
    fitted to the last 3 available 5-year data points.
    
    Returns the full DataFrame with extrapolated rows appended
    and a boolean column 'is_extrapolated'.
    """
    # Mark existing data as not extrapolated
    df["is_extrapolated"] = False
    
    # Years to extrapolate
    extra_years = [y for y in FIVE_YEAR_STEPS if y > max_year_in_data and y <= target_max_year]
    if not extra_years:
        print(f"  No extrapolation needed for {var_name}")
        return df
    
    print(f"  Extrapolating {var_name}: {extra_years[0]}–{extra_years[-1]}")
    
    new_rows = []
    for (country, scenario), group in df.groupby(["country_name", "scenario"]):
        group_sorted = group.sort_values("year")
        
        # Last 3 data points for trend
        tail = group_sorted.tail(3)
        if len(tail) < 2:
            continue
        
        # Fit linear trend: value = slope * year + intercept
        x = tail["year"].values.astype(float)
        y = tail[var_name].values.astype(float)
        if np.any(np.isnan(y)):
            continue
        
        slope = np.polyfit(x, y, 1)[0]
        last_year = tail["year"].iloc[-1]
        last_val = tail[var_name].iloc[-1]
        
        for yr in extra_years:
            projected = last_val + slope * (yr - last_year)
            projected = np.clip(projected, clip_min, clip_max)
            new_rows.append({
                "country_name": country,
                "scenario": scenario,
                "year": yr,
                var_name: projected,
                "is_extrapolated": True
            })
    
    extra_df = pd.DataFrame(new_rows)
    result = pd.concat([df, extra_df], ignore_index=True)
    print(f"  Added {len(extra_df)} extrapolated rows")
    return result

In [ ]:
# Extrapolate Employment in Agriculture (2050 → 2100, clip 0–100%)
agri_fc = extrapolate_linear(
    agri_fc, "employment_agriculture",
    max_year_in_data=2050, target_max_year=2100,
    clip_min=0, clip_max=100
)
print(f"Agriculture forecast years: {sorted(agri_fc['year'].unique())}")
print()

In [ ]:
# Extrapolate HDI (2075 → 2100, clip 0–1)
hdi_fc = extrapolate_linear(
    hdi_fc, "hdi",
    max_year_in_data=2075, target_max_year=2100,
    clip_min=0, clip_max=1
)
print(f"HDI forecast years: {sorted(hdi_fc['year'].unique())}")

## 4. Harmonize country names across sources

The SSP Excel uses the reference country names. For each Explorer file,  
we check which names don't match and map them manually.

In [ ]:
# Reference country names from the SSP Excel
excel_countries = set(ssp_pivot["country_name"].unique())
print(f"SSP Excel countries: {len(excel_countries)}")

# Check each Explorer dataset for mismatches
for name, df in [("Corruption", corruption_fc), ("Agriculture", agri_fc),
                  ("Gini", gini_fc), ("HDI", hdi_fc)]:
    explorer_countries = set(df["country_name"].unique())
    in_explorer_not_excel = explorer_countries - excel_countries
    in_excel_not_explorer = excel_countries - explorer_countries
    print(f"\n{name}:")
    print(f"  Countries: {len(explorer_countries)}")
    if in_explorer_not_excel:
        print(f"  In Explorer but not Excel ({len(in_explorer_not_excel)}): {sorted(in_explorer_not_excel)[:10]}")
    if in_excel_not_explorer:
        print(f"  In Excel but not Explorer ({len(in_excel_not_explorer)}): {sorted(in_excel_not_explorer)[:10]}")

In [ ]:
# Mapping: Explorer name → Excel name (if different)
# Add entries here based on the mismatches printed above
EXPLORER_TO_EXCEL = {
    "Bolivia (Plurinational State of)": "Bolivia",
    "Cabo Verde": "Cape Verde",
    "China, Hong Kong SAR": "Hong Kong",
    "China, Macao SAR": "Macao",
    "China, Taiwan Province of China": "Taiwan",
    "Congo": "Republic of the Congo",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Czechia": "Czech Republic",
    "Democratic People's Republic of Korea": "North Korea",
    "Democratic Republic of the Congo": "Democratic Republic of the Congo",
    "Eswatini": "Swaziland",
    "Iran (Islamic Republic of)": "Iran",
    "Lao People's Democratic Republic": "Laos",
    "Micronesia (Federated States of)": "Micronesia",
    "Moldova (Republic of)": "Moldova",
    "North Macedonia": "TFYR Macedonia",
    "Republic of Korea": "South Korea",
    "Republic of the Congo": "Republic of the Congo",
    "Russian Federation": "Russia",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "State of Palestine": "Palestine",
    "Syrian Arab Republic": "Syria",
    "Tanzania (United Republic of)": "Tanzania",
    "Timor-Leste": "East Timor",
    "Türkiye": "Turkey",
    "United Kingdom": "United Kingdom",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Viet Nam": "Vietnam",
}

def harmonize_names(df):
    """Apply name mapping so Explorer names match SSP Excel names."""
    df["country_name"] = df["country_name"].replace(EXPLORER_TO_EXCEL)
    return df

corruption_fc = harmonize_names(corruption_fc)
agri_fc = harmonize_names(agri_fc)
gini_fc = harmonize_names(gini_fc)
hdi_fc = harmonize_names(hdi_fc)

print("Country names harmonized.")

### Verify harmonization: re-check mismatches

In [ ]:
for name, df in [("Corruption", corruption_fc), ("Agriculture", agri_fc),
                  ("Gini", gini_fc), ("HDI", hdi_fc)]:
    explorer_countries = set(df["country_name"].unique())
    still_missing = explorer_countries - excel_countries
    if still_missing:
        print(f"{name} — still unmatched: {sorted(still_missing)[:10]}")
    else:
        print(f"{name} — all matched ✓")

> **Note:** If any names are still unmatched after running the cell above, add them to the
> `EXPLORER_TO_EXCEL` dictionary and re-run from the harmonization cell.

## 5. Add country codes

We use the historical panel's name→code mapping built from World Bank data.  
We also build a mapping from SSP Excel names → WB codes via the SSP name fixes from notebook 01.

In [ ]:
# Load the historical panel to get the name→code mapping
hist = pd.read_csv(DATA_PROCESSED / "historical_panel.csv")
wb_name_to_code = (
    hist[["country_name", "country_code"]]
    .drop_duplicates()
    .set_index("country_name")["country_code"]
    .to_dict()
)
print(f"WB name→code mapping: {len(wb_name_to_code)} entries")

# SSP Excel name → WB name mapping (reverse of what we used in notebook 01)
SSP_EXCEL_TO_WB = {
    "Bolivia": "Bolivia",
    "Cape Verde": "Cabo Verde",
    "Cote d'Ivoire": "Cote d'Ivoire",
    "Czech Republic": "Czechia",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "East Timor": "Timor-Leste",
    "Egypt": "Egypt, Arab Rep.",
    "Gambia": "Gambia, The",
    "Hong Kong": "Hong Kong SAR, China",
    "Iran": "Iran, Islamic Rep.",
    "Laos": "Lao PDR",
    "Macao": "Macao SAR, China",
    "Micronesia": "Micronesia, Fed. Sts.",
    "North Korea": "Korea, Dem. People's Rep.",
    "Palestine": "West Bank and Gaza",
    "Republic of the Congo": "Congo, Rep.",
    "Russia": "Russian Federation",
    "Slovakia": "Slovak Republic",
    "South Korea": "Korea, Rep.",
    "St. Kitts and Nevis": "St. Kitts and Nevis",
    "St. Lucia": "St. Lucia",
    "St. Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Swaziland": "Eswatini",
    "Syria": "Syrian Arab Republic",
    "TFYR Macedonia": "North Macedonia",
    "Taiwan": "Taiwan, China",
    "Turkey": "Turkiye",
    "United States": "United States",
    "Venezuela": "Venezuela, RB",
    "Vietnam": "Vietnam",
    "Yemen": "Yemen, Rep.",
}

def add_country_code(df):
    """Map country_name → country_code using WB name mapping."""
    # Try direct WB match first, then try via SSP→WB mapping
    df["wb_name"] = df["country_name"].map(lambda x: SSP_EXCEL_TO_WB.get(x, x))
    df["country_code"] = df["wb_name"].map(wb_name_to_code)
    
    unmatched = df[df["country_code"].isna()]["country_name"].unique()
    if len(unmatched) > 0:
        print(f"  Could not map {len(unmatched)} countries to WB codes: {sorted(unmatched)[:10]}")
    
    df = df.drop(columns=["wb_name"])
    return df

In [ ]:
print("Adding country codes...")
print("SSP GDP/Pop:")
ssp_pivot = add_country_code(ssp_pivot)
print("Corruption:")
corruption_fc = add_country_code(corruption_fc)
print("Agriculture:")
agri_fc = add_country_code(agri_fc)
print("Gini:")
gini_fc = add_country_code(gini_fc)
print("HDI:")
hdi_fc = add_country_code(hdi_fc)

## 6. Merge all forecast features

Join on `(country_name, scenario, year)`. Start with the GDP/Population panel as the base.

In [ ]:
# Base: SSP GDP/Pop
forecast = ssp_pivot[["country_name", "country_code", "scenario", "year",
                       "gdp_per_capita", "population"]].copy()

print(f"Base panel: {forecast.shape}")

# Merge each Explorer forecast
merge_keys = ["country_name", "scenario", "year"]

for df, var_col in [
    (corruption_fc, "control_of_corruption"),
    (agri_fc, "employment_agriculture"),
    (gini_fc, "gini"),
    (hdi_fc, "hdi"),
]:
    # Keep only the merge keys + value column (drop is_extrapolated for merge)
    cols_to_merge = merge_keys + [var_col]
    if "is_extrapolated" in df.columns:
        cols_to_merge.append("is_extrapolated")
    
    merge_df = df[cols_to_merge].copy()
    
    # Rename is_extrapolated to avoid conflicts
    if "is_extrapolated" in merge_df.columns:
        merge_df = merge_df.rename(columns={"is_extrapolated": f"is_extrapolated_{var_col}"})
    
    forecast = forecast.merge(merge_df, on=merge_keys, how="left")
    matched = forecast[var_col].notna().sum()
    print(f"  Merged {var_col}: {matched}/{len(forecast)} non-null")

print(f"\nFinal forecast panel: {forecast.shape}")
forecast.head()

### Consolidate extrapolation flags

In [ ]:
# Create a single is_extrapolated column: True if ANY feature was extrapolated
extrap_cols = [c for c in forecast.columns if c.startswith("is_extrapolated_")]
if extrap_cols:
    forecast["is_extrapolated"] = forecast[extrap_cols].any(axis=1).fillna(False)
    forecast = forecast.drop(columns=extrap_cols)
else:
    forecast["is_extrapolated"] = False

print(f"Extrapolated rows: {forecast['is_extrapolated'].sum()} / {len(forecast)}")

## 7. Final checks and save

In [ ]:
# Final column order
output_cols = [
    "country_name", "country_code", "scenario", "year",
    "gdp_per_capita", "population", "hdi",
    "control_of_corruption", "employment_agriculture", "gini",
    "is_extrapolated"
]
forecast = forecast[output_cols].sort_values(
    ["scenario", "country_name", "year"]
).reset_index(drop=True)

print("=" * 60)
print("SSP FORECAST PANEL SUMMARY")
print("=" * 60)
print(f"Shape:      {forecast.shape}")
print(f"Countries:  {forecast['country_code'].nunique()}")
print(f"Scenarios:  {sorted(forecast['scenario'].unique())}")
print(f"Year range: {forecast['year'].min()} – {forecast['year'].max()}")
print(f"Year steps: {sorted(forecast['year'].unique())}")
print()
print("% NaN per variable:")
var_cols = ["gdp_per_capita", "population", "hdi", "control_of_corruption",
            "employment_agriculture", "gini"]
print((forecast[var_cols].isna().mean() * 100).round(2))
print()
print(forecast.describe().round(3))

In [ ]:
forecast.to_csv(DATA_PROCESSED / "ssp_forecast_panel.csv", index=False)
print(f"Saved: {DATA_PROCESSED / 'ssp_forecast_panel.csv'}")

---
**Output produced:** `data/processed/ssp_forecast_panel.csv`  
Contains one row per country × scenario × 5-year step (2025–2100) with 5 features + extrapolation flag.